# Five tables, and two kinds of separator

The tables live in `data/`. `u.info` is the three-line contents label: 943 users, 1682 items, 100000 ratings.

| File | What a row is | Separator |
|---|---|---|
| `u.data` | user id, movie id, rating, timestamp | tab |
| `u.user` | user id, age, gender, occupation, zip code | pipe |
| `u.item` | movie id, title, release date, video release date, IMDb URL, then 19 genre flags | pipe |
| `u.genre` | genre name and its column number, from 0 to 18 | pipe |
| `u.occupation` | one occupation name per line | none |

The README calls `u.item` tab-separated. The file itself uses pipes. Splitting that file on tabs returns one field per movie.

A movie row has 24 fields. After the URL come the 19 genre flags, in the order `u.genre` lists: unknown, Action, Adventure, Animation, Children's, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western. A 1 means the movie carries that genre. A movie may carry several.

The video-release field is empty on every movie: the row has a pipe, then nothing, then a pipe. That empty field is not the title and not a genre.

`ua.base` and `ua.test` are the split used later. The test file holds 10 ratings for every user, $943 \times 10 = 9430$ ratings. The base file holds the other $100000 - 9430 = 90570$. The same pair of ratings never appears in both files. `ub.base` and `ub.test` are a second split of the same kind, and the two test files do not share a rating. `u1` through `u5` are five disjoint 20000-rating test folds, with 80000-rating training files.

Open every text file as Latin-1. Nine movie titles contain characters outside ASCII. Reading the file as UTF-8 breaks those titles.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# Separators, field counts, and the official split sizes.
info = read_lines("u.info")
print(info)
assert info == ["943 users", "1682 items", "100000 ratings"]
users = [line.split("|") for line in read_lines("u.user")]
movies = [line.split("|") for line in read_lines("u.item")]
genres = [line.split("|")[0] for line in read_lines("u.genre") if line.strip()]
jobs = [line for line in read_lines("u.occupation") if line.strip()]
print(len(users), len(movies), len(genres), len(jobs))
print(len(movies[0]), movies[0][1])
assert len(users) == 943 and all(len(row) == 5 for row in users)
assert len(movies) == 1682 and all(len(row) == 24 for row in movies)
assert movies[0][1] == "Toy Story (1995)"
assert movies[0][3] == ""
assert sum(row[3] == "" for row in movies) == 1682
assert len(genres) == 19 and genres[0] == "unknown" and genres[-1] == "Western"
assert len(jobs) == 21
base = read_ratings("ua.base")
test = read_ratings("ua.test")
other = read_ratings("ub.test")
print(len(base), len(test), len(other))
assert (len(base), len(test)) == (90570, 9430)
assert len({(user, item) for user, item, _, _ in ratings}) == 100000
base_keys = {(user, item) for user, item, _, _ in base}
test_keys = {(user, item) for user, item, _, _ in test}
other_keys = {(user, item) for user, item, _, _ in other}
assert base_keys.isdisjoint(test_keys)
assert test_keys.isdisjoint(other_keys)
assert base_keys | test_keys == {(user, item) for user, item, _, _ in ratings}
odd_titles = [
    row[1] for row in movies if any(ord(character) > 127 for character in row[1])
]
print("titles outside ASCII", len(odd_titles))
assert len(odd_titles) == 9


['943 users', '1682 items', '100000 ratings']
943 1682 19 21
24 Toy Story (1995)
90570 9430 9430
titles outside ASCII 9


## Pitfall

The README and the movie file disagree about the separator. The bytes decide: `u.data` is tabs, `u.item` and `u.user` are pipes. A tab split of a movie row hides the genres inside one string.

## Summary

943 users, 1682 movies, 100000 ratings. Genre flags are 19 zeros and ones after the URL. The `ua` test file holds exactly 9430 ratings, ten per user, and shares none of them with `ua.base`.
